# Forward sensitivity analysis (FSA): formulation and validation

The ChemKAN paper trains with forward sensitivity analysis (p. 8): the model's ODE and its
sensitivity equations are solved together with Tsit5, and the gradient of the loss is
assembled from the sensitivities instead of backpropagating through the solver. In this
repository every trainer takes `--sensitivity fsa` (this method, `chemkan/src/chemkan/fsa.py`)
or `--sensitivity direct_autograd` (backpropagation through the solver's operations).

| recorded runs | gradient backend |
|---|---|
| biodiesel ChemKAN figure runs (Figs. 3–6), `biodiesel/run_figure_runs.py` | `fsa` |
| hydrogen and methane main runs, `combustion/run_hydrogen.sh`, `run_methane.sh` | `direct_autograd` |

This notebook

1. states the sensitivity equations and maps each term onto `fsa.py`;
2. checks the machinery on an ODE with a known solution;
3. checks that FSA differentiates exactly the right-hand side the trainers integrate;
4. compares FSA and direct-autograd gradients on the same parameters, data, loss and solver
   settings, for the final biodiesel model and hydrogen Stage 1 and Stage 2, with the
   committed checkpoints;
5. shows how the agreement depends on the solver tolerance;
6. checks that the trajectories of a batch stay independent and that the augmented solve
   reproduces the ordinary state solve;
7. runs the FSA unit tests (`chemkan/tests/test_fsa.py`);
8. summarizes the earlier validation record kept on the development branch (not rerun here).

Everything runs in float64 on the CPU, about three minutes on a laptop.

In [1]:
from pathlib import Path
import subprocess, sys, time

import numpy as np
import pandas as pd
import torch
from torch.func import jacrev

ROOT = Path.cwd()
while not (ROOT / "chemkan" / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in ("chemkan/src", "chemkan/scripts", "chemkan/scripts/combustion"):
    sys.path.insert(0, str(ROOT / p))
torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
pd.options.display.float_format = "{:.2e}".format
pd.options.display.width = 250
pd.options.display.max_columns = None

from chemkan.fsa import FunctionalDynamics, integrate_with_sensitivities
from chemkan.solver import SolverConfig, integrate
from chemkan.training import loss_and_gradients
import train_biodiesel as bd                     # the biodiesel trainer: data loading
from reference_components import NormalizedDynamics, build_core, julia_mse
import train_combustion as tc                    # the hydrogen / methane trainer

LEVELS = [(1e-6, 1e-8), (1e-8, 1e-10), (1e-10, 1e-12)]   # (rtol, atol)


def solver(rtol, atol, backend):
    return SolverConfig(method="tsit5", rtol=rtol, atol=atol, sensitivity=backend)


def rel(a, b):
    return float((a - b).norm() / b.norm())


def gradient(func, y0, t, params, loss_fn, cfg):
    # One gradient evaluation through chemkan.training.loss_and_gradients, the call both
    # backends share in the trainers; returns (flat gradient, loss, seconds).
    for p in params:
        p.grad = None
    t0 = time.perf_counter()
    out = loss_and_gradients(func, y0, t, params, loss_fn, cfg)
    loss = out[0] if isinstance(out, tuple) else out
    g = torch.cat([p.grad.reshape(-1) for p in params]).clone()
    return g, float(loss.detach()), time.perf_counter() - t0

## 1. The sensitivity equations and their implementation

Each trainer integrates a stage's state $x$ in normalized coordinates:

| stage | state $x$ | right-hand side $F(x, t; \theta)$ | parameters $\theta$ |
|---|---|---|---|
| biodiesel | $z = [\hat Y, \hat T]$ | $[\mathrm{KAN_{kin}}(z), 0] \,/\, 50\ \mathrm{s}$ | $\theta_\mathrm{kin}$ (156) |
| hydrogen / methane Stage 1 | $\hat Y$ | $\mathrm{KAN_{kin}}([\hat Y, \hat T_\mathrm{obs}(t)]) \,/\, t_\mathrm{ref}$ | $\theta_\mathrm{kin}$ (285 for hydrogen) |
| hydrogen / methane Stage 2 | $\hat u = [\hat Y, \hat T]$ | $[r, \mathrm{Linear}(r) + \mathrm{KAN_{cor}}(\hat u)] \,/\, t_\mathrm{ref}$, $r = \mathrm{KAN_{kin}}(\hat u)$ | $\theta_\mathrm{kin}, \theta_\mathrm{thermo}, \theta_\mathrm{cor}$ (344 for hydrogen) |

These are the paper's Eqs. 13 and 15 in the normalized coordinates of the authors' released
example. The initial state $x(t_0)$ is observed data and does not depend on $\theta$.
Differentiating $\mathrm{d}x/\mathrm{d}t = F(x, t; \theta)$ with respect to $\theta$ gives the
forward sensitivity equations for $S = \partial x / \partial \theta$ (one column per
parameter):

$$
\frac{\mathrm{d}S}{\mathrm{d}t} = \frac{\partial F}{\partial x}\, S + \frac{\partial F}{\partial \theta},
\qquad S(t_0) = 0 .
$$

For a loss $L$ of the predicted states at the observation times $t_1, \dots, t_N$, the chain
rule assembles the gradient from the sensitivities:

$$
\frac{\mathrm{d}L}{\mathrm{d}\theta} = \sum_{j=1}^{N} S(t_j)^\top \, \frac{\partial L}{\partial x(t_j)} .
$$

In biodiesel, $\mathrm{d}\hat T/\mathrm{d}t = 0$, so the temperature row of $S$ stays zero. In
Stage 1 the temperature is an input, not a state; its time dependence enters through
$F(x, t)$. The formulation is the standard continuous forward sensitivity analysis
([SciMLSensitivity.jl, "Sensitivity Math Details"](https://docs.sciml.ai/SciMLSensitivity/stable/sensitivity_math/)).

| mathematics | `chemkan/src/chemkan/fsa.py` |
|---|---|
| $F(x_b, t; \theta)$ of one trajectory, through the trainer's own dynamics module | `FunctionalDynamics._single` (`torch.func.functional_call`), `rhs` |
| $\partial F/\partial x$ and $\partial F/\partial \theta$ per trajectory | `FunctionalDynamics.rhs_and_jacobians` (`jacrev`, `vmap` over trajectories) |
| augmented state $[x, S]$ with $\mathrm{d}S/\mathrm{d}t = (\partial F/\partial x) S + \partial F/\partial\theta$ | `AugmentedSensitivityRHS` (`torch.baddbmm`) |
| one Tsit5 solve of the augmented state, $S(t_0) = 0$, no autograd graph | `integrate_with_sensitivities` |
| $\partial L/\partial x(t_j)$ from the unchanged loss on a detached copy of the prediction | `fsa_loss_and_gradients`, step 2 |
| $\sum_j S(t_j)^\top \partial L/\partial x(t_j)$ | `contract_sensitivities` (`einsum("tbnp,tbn->p")`) |
| parameter order and the groups $\theta_\mathrm{kin}$, $\theta_\mathrm{thermo}$, $\theta_\mathrm{cor}$ | `ParameterPacking` |

The solver controls the error of the whole augmented state with the same `rtol`/`atol` and
torchdiffeq's RMS norm. FSA and direct autograd therefore take different adaptive steps: they
agree to the solver's accuracy, not bit for bit.

## 2. Analytic check: $\mathrm{d}y/\mathrm{d}t = \theta y$

Not a ChemKAN problem: it tests the machinery on a case with a known answer,
$y = y_0 e^{\theta t}$, $S = \partial y/\partial\theta = t\,y(t)$. For
$L = \sum_j (y(t_j) - c_j)^2$ the exact gradient is $\sum_j 2\,(y(t_j) - c_j)\, t_j\, y(t_j)$.
The errors should fall as the tolerance tightens.

In [2]:
class Exponential(torch.nn.Module):
    def __init__(self, theta):
        super().__init__()
        self.theta = torch.nn.Parameter(torch.tensor(theta))

    def forward(self, t, y):
        return self.theta * y


theta, y0 = -0.7, torch.tensor([[1.3]])
t = torch.linspace(0.0, 2.0, 9)
dyn = Exponential(theta)
y_exact = y0[0, 0] * torch.exp(theta * t)
S_exact = t * y_exact
target = 0.9 * y_exact                                    # any fixed observations c_j
g_exact = float((2 * (y_exact - target) * S_exact).sum())
loss_fn = lambda pred: ((pred[:, 0, 0] - target) ** 2).sum()

rows = []
for rtol, atol in LEVELS + [(1e-12, 1e-14)]:
    y, S = integrate_with_sensitivities(FunctionalDynamics(dyn, [dyn.theta]), y0, t,
                                        solver(rtol, atol, "fsa"))
    g, _, _ = gradient(dyn, y0, t, [dyn.theta], loss_fn, solver(rtol, atol, "fsa"))
    rows.append({"rtol / atol": f"{rtol:g} / {atol:g}",
                 "state rel. error": float((y[:, 0, 0] - y_exact).abs().max() / y_exact.abs().max()),
                 "sensitivity rel. error": float((S[:, 0, 0, 0] - S_exact).abs().max() / S_exact.abs().max()),
                 "dL/dθ rel. error": abs(float(g[0]) - g_exact) / abs(g_exact)})
pd.DataFrame(rows).set_index("rtol / atol")

,state rel. error,sensitivity rel. error,dL/dθ rel. error
rtol / atol,,,
1e-06 / 1e-08,2.00e-07,2.10e-06,1.48e-06
1e-08 / 1e-10,5.61e-09,5.66e-08,4.17e-08
1e-10 / 1e-12,1.46e-10,1.44e-09,1.10e-09
1e-12 / 1e-14,3.68e-12,3.63e-11,2.81e-11


## 3. FSA differentiates the right-hand side the trainers integrate

FSA does not use a second implementation of ChemKAN: `FunctionalDynamics` evaluates the
trainer's own dynamics module through `functional_call`, with the parameters taken from one
flat vector. Its output must equal the module's output. Checked here on the final
configurations, with the committed weights:

- biodiesel: the 5 % noise figure run (Fig. 5), `NormalizedDynamics` with the divisor 50 s;
- hydrogen Stage 1: the 100,000-epoch Stage-1 archive, `Stage1Dynamics` with the dense observed temperature;
- hydrogen Stage 2: the 200,000-epoch Stage-2 checkpoint, `Stage2Dynamics`.

The hydrogen cases use two training conditions, 1050 K / φ = 0.9 and 1200 K / φ = 0.7.

In [3]:
# biodiesel: the trainer's data loader and the figure run's weights
BD_RUN = ROOT / "results/biodiesel/chemkan_figure_runs/fig05/chemkan_noise05_seed0"
bd_data = bd.load_canonical_data(torch.device("cpu"), ROOT / "chemkan/data/generated/biodiesel_v2.npz", 5)
core = build_core(4)
core.load_state_dict(torch.load(BD_RUN / "checkpoint_final.pt", weights_only=False)["model_state"])
bd_dyn = NormalizedDynamics(core, 50.0)
bd_x0 = bd_data["u0"][bd_data["train"]]                           # (20, 7)
bd_target = bd_data["target"][bd_data["train"]]                   # (20, 6, 30) noisy observations
bd_t = bd_data["t"]
bd_loss = lambda pred: julia_mse(pred[:, :, :6].permute(1, 2, 0), bd_target)   # the trainer's loss

# hydrogen: the trainer's problem (training split, dense observed temperature), two conditions
tc.configure_system("hydrogen")
problem = tc.load_problem(torch.float64)
IDX = [14, 30]
sub = {**problem, "u_hat": problem["u_hat"][:, IDX], "T_dense_hat": problem["T_dense_hat"][:, IDX]}
print("hydrogen conditions (T0 [K], phi):", np.load(ROOT / "chemkan/data/generated/hydrogen.npz")["train_ics"][IDX].tolist())


def hydrogen_case(stage, checkpoint):
    model = tc.build_model().double()
    model.load_state_dict(torch.load(ROOT / checkpoint, weights_only=False)["model_state"])
    units = (tc.stage1_units if stage == 1 else tc.stage2_units)(model, sub, tc.T_REF_DEFAULT, "batched")
    func, y0, loss_fn = units[0]                                  # batched: one unit
    params = list(model.kinetic.parameters()) if stage == 1 else list(model.parameters())
    return {"func": func, "x0": y0, "t": sub["t"], "loss": loss_fn, "params": params}


CASES = {
    "biodiesel (Fig. 5, 5 % noise)": {"func": bd_dyn, "x0": bd_x0, "t": bd_t, "loss": bd_loss,
                                      "params": list(core.parameters())},
    "hydrogen Stage 1 (100k)": hydrogen_case(1, "results/hydrogen/S1_seed0/archive/stage1_S1_seed0_epoch100000.pt"),
    "hydrogen Stage 2 (200k)": hydrogen_case(2, "results/hydrogen/S2_seed0_200k/stage2_final.pt"),
}
rows = []
for name, c in CASES.items():
    fd = FunctionalDynamics(c["func"], c["params"])
    with torch.no_grad():
        for tj in (c["t"][0], c["t"][len(c["t"]) // 2], c["t"][-1]):
            F_module, F_functional = c["func"](tj, c["x0"]), fd.rhs(tj, c["x0"])
            rows.append({"case": name, "t [s]": f"{float(tj):.3g}", "parameters": fd.packing.total,
                         "max |difference| / max |F|": float((F_functional - F_module).abs().max()
                                                             / F_module.abs().max())})
pd.DataFrame(rows).set_index(["case", "t [s]"])

hydrogen conditions (T0 [K], phi): [[1050.0, 0.9], [1200.0, 0.7]]


parameters  max |difference| / max |F|
case                          t [s]                                           
biodiesel (Fig. 5, 5 % noise) 0                156                    1.89e-16
                              15.5             156                    1.89e-16
                              30               156                    1.89e-16
hydrogen Stage 1 (100k)       0                285                    0.00e+00
                              0.000303         285                    0.00e+00
                              0.0006           285                    0.00e+00
hydrogen Stage 2 (200k)       0                344                    1.35e-17
                              0.000303         344                    1.35e-17
                              0.0006           344                    1.35e-17

## 4. FSA vs direct autograd: same parameters, data, loss and solver

For each case, both backends compute the gradient of the trainer's loss at the same weights
with the same solver settings, through `chemkan.training.loss_and_gradients`. The table gives
the relative L2 difference of the full gradient and of each parameter group, the losses, and
the time per gradient evaluation (one thread). The first row of each case is the setting of
the recorded runs:

- biodiesel: rtol 1e-2, atol 1e-6 (the released example's training tolerances), then tighter;
- hydrogen: rtol 1e-6, atol 1e-8 (the trainer's default), then tighter.

In [4]:
LEVELS_BY_CASE = {name: ([(1e-2, 1e-6)] if name.startswith("biodiesel") else []) + LEVELS for name in CASES}
# fsa.ParameterPacking groups; biodiesel's core is entirely the kinetic core ("theta")
GROUP = {"theta_kin": "θ_kin", "theta": "θ_kin", "theta_thermo": "θ_thermo", "theta_cor": "θ_cor"}
grads, rows = {}, []
for name, c in CASES.items():
    packing = FunctionalDynamics(c["func"], c["params"]).packing
    groups = {GROUP[g]: ix for g, ix in packing.group_indices().items()}
    for rtol, atol in LEVELS_BY_CASE[name]:
        g_da, L_da, s_da = gradient(c["func"], c["x0"], c["t"], c["params"], c["loss"], solver(rtol, atol, "direct_autograd"))
        g_fsa, L_fsa, s_fsa = gradient(c["func"], c["x0"], c["t"], c["params"], c["loss"], solver(rtol, atol, "fsa"))
        grads[name, rtol] = (g_da, g_fsa)
        row = {"case": name, "rtol / atol": f"{rtol:g} / {atol:g}", "all parameters": rel(g_fsa, g_da)}
        row.update({grp: rel(g_fsa[ix], g_da[ix]) for grp, ix in groups.items()})
        row.update({"loss DA": L_da, "loss FSA": L_fsa, "s DA / FSA": f"{s_da:.1f} / {s_fsa:.1f}"})
        rows.append(row)
COLUMNS = ["all parameters", "θ_kin", "θ_thermo", "θ_cor", "loss DA", "loss FSA", "s DA / FSA"]
pd.DataFrame(rows).set_index(["case", "rtol / atol"])[COLUMNS].fillna("—")

all parameters    θ_kin θ_thermo    θ_cor  loss DA  loss FSA   s DA / FSA
case                          rtol / atol                                                                             
biodiesel (Fig. 5, 5 % noise) 0.01 / 1e-06         9.68e-01 9.68e-01        —        — 1.12e-03  1.12e-03    0.0 / 0.1
                              1e-06 / 1e-08        4.26e-03 4.26e-03        —        — 1.12e-03  1.12e-03    0.1 / 0.2
                              1e-08 / 1e-10        1.23e-04 1.23e-04        —        — 1.12e-03  1.12e-03    0.1 / 0.5
                              1e-10 / 1e-12        3.68e-06 3.68e-06        —        — 1.12e-03  1.12e-03    0.3 / 1.2
hydrogen Stage 1 (100k)       1e-06 / 1e-08        5.16e-05 5.16e-05        —        — 9.13e-01  9.13e-01    0.4 / 1.9
                              1e-08 / 1e-10        2.38e-05 2.38e-05        —        — 9.13e-01  9.13e-01    1.3 / 5.7
                              1e-10 / 1e-12        2.68e-06 2.68e-06        —        — 9.13e-01  9.13e-01  11.3 / 44.0
hydrogen Stage 2 (200k)       1e-06 / 1e-08        2.62e-04 2.44e-04 2.52e-04 3.54e-04 4.09e-02  4.09e-02    0.6 / 3.3
                              1e-08 / 1e-10        5.33e-06 5.02e-06 5.08e-06 6.98e-06 4.09e-02  4.09e-02    1.3 / 7.2
                              1e-10 / 1e-12        1.50e-07 1.45e-07 1.35e-07 1.82e-07 4.09e-02  4.09e-02   3.2 / 16.8

## 5. Tolerance sensitivity and numerical error

Both backends approximate the same exact gradient, with an error set by the solver
tolerance: FSA integrates the continuous sensitivity equations, direct autograd differentiates
the solver's discrete steps. Using each case's tightest level (rtol 1e-10) as the reference,
the table shows how far each backend's gradient at the coarser levels is from it, next to the
FSA-vs-autograd difference of section 4. As the tolerance tightens, the deviations shrink and
the two backends converge to the same gradient. Hydrogen Stage 1 converges slowly: its
direct-autograd gradient is not closer to the reference at rtol 1e-8 than at 1e-6 (the
development-branch record, section 8, found the same for Stage 1 and resolved it one tolerance
decade further).

The biodiesel row at rtol 1e-2 / atol 1e-6, the tolerances of the released example and of the
recorded biodiesel figure runs, is different. There, neither backend is close to the converged
gradient (both deviations are of order one or larger), and the two differ from each other by
about as much. The loss itself is accurate at that tolerance (section 4: the same value to three digits
at every level); its gradient is not.

In [5]:
rows = []
for name in CASES:
    levels = LEVELS_BY_CASE[name]
    ref_da, ref_fsa = grads[name, levels[-1][0]]
    for rtol, _ in levels[:-1]:
        g_da, g_fsa = grads[name, rtol]
        rows.append({"case": name, "rtol": f"{rtol:g}",
                     "DA vs DA at 1e-10": rel(g_da, ref_da), "FSA vs FSA at 1e-10": rel(g_fsa, ref_fsa),
                     "FSA vs DA (same rtol)": rel(g_fsa, g_da)})
pd.DataFrame(rows).set_index(["case", "rtol"])

DA vs DA at 1e-10  FSA vs FSA at 1e-10  FSA vs DA (same rtol)
case                          rtol                                                                
biodiesel (Fig. 5, 5 % noise) 0.01            2.33e+01             1.71e+00               9.68e-01
                              1e-06           6.55e-03             2.44e-03               4.26e-03
                              1e-08           1.86e-04             6.90e-05               1.23e-04
hydrogen Stage 1 (100k)       1e-06           1.66e-05             5.50e-05               5.16e-05
                              1e-08           2.21e-05             4.49e-06               2.38e-05
hydrogen Stage 2 (200k)       1e-06           2.69e-04             1.82e-05               2.62e-04
                              1e-08           5.60e-06             4.99e-07               5.33e-06

## 6. Batched trajectories stay independent; the augmented solve keeps the state

A batch shares the parameters, but its trajectories are separate ODEs: the rate of trajectory
$b$ must not depend on the state of trajectory $c$, so the cross-trajectory blocks of
$\partial F/\partial x$ must be exactly zero. Then the batched augmented solve must give the
same states and sensitivities as solving each trajectory alone (to the solver's accuracy:
a batch shares its adaptive steps), and its state part must equal the ordinary state-only solve
the trainers use for direct autograd and for evaluation.

In [6]:
from chemkan.temperature import ObservedTemperature


def alone(name, c, b):
    # The case restricted to trajectory b. Stage 1 needs trajectory b's own observed
    # temperature; the biodiesel and Stage-2 dynamics carry no per-trajectory data.
    if name.startswith("hydrogen Stage 1"):
        temp = ObservedTemperature(sub["t_dense"], sub["T_dense_hat"][:, b:b + 1])
        return tc.Stage1Dynamics(c["func"].kinetic, temp, tc.T_REF_DEFAULT), c["x0"][b:b + 1]
    return c["func"], c["x0"][b:b + 1]


cfg = solver(1e-10, 1e-12, "fsa")
rows = []
for name, c in CASES.items():
    x0 = c["x0"][:3]                                         # biodiesel: 3 of the 20 trajectories
    fd = FunctionalDynamics(c["func"], c["params"])
    J = jacrev(lambda x, fd=fd, t=c["t"][1]: fd.rhs(t, x))(x0)   # (B, n, B, n)
    B = x0.shape[0]
    cross = max(float(J[i, :, j, :].abs().max()) for i in range(B) for j in range(B) if i != j)
    x, S = integrate_with_sensitivities(fd, x0, c["t"], cfg)
    b = 1
    func_b, x0_b = alone(name, c, b)
    x_b, S_b = integrate_with_sensitivities(FunctionalDynamics(func_b, c["params"]), x0_b, c["t"], cfg)
    with torch.no_grad():
        x_state = integrate(c["func"], x0, c["t"], cfg)
    rows.append({"case": name, "trajectories": B, "max cross-trajectory |∂F_b/∂x_c|": cross,
                 "batched vs alone: state": float((x[:, b] - x_b[:, 0]).abs().max() / x.abs().max()),
                 "batched vs alone: S": float((S[:, b] - S_b[:, 0]).abs().max() / S.abs().max()),
                 "augmented vs state-only solve": float((x - x_state).abs().max() / x.abs().max())})
pd.DataFrame(rows).set_index("case")

,trajectories,max cross-trajectory |∂F_b/∂x_c|,batched vs alone: state,batched vs alone: S,augmented vs state-only solve
case,,,,,
"biodiesel (Fig. 5, 5 % noise)",3,0.00e+00,9.15e-11,1.70e-10,1.63e-09
hydrogen Stage 1 (100k),2,0.00e+00,5.99e-08,8.20e-07,6.86e-08
hydrogen Stage 2 (200k),2,0.00e+00,2.90e-10,2.20e-10,6.87e-09


## 7. The FSA unit tests

`chemkan/tests/test_fsa.py` checks the same properties on small ChemKAN problems (biodiesel,
Stage 1, Stage 2 with random weights): the analytic problem, parameter packing and groups,
functional equivalence, trajectory independence (also under a change of one trajectory's
observed temperature), FSA vs direct autograd at rtol 1e-10 (relative difference below 1e-6),
batched vs individual solves, the augmented vs state-only solve, explicit zero gradients,
the abort on non-finite values, and that a resumed run refuses a change of backend.

In [7]:
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_fsa.py", "-q", "-p", "no:cacheprovider", "--color=no"],
                        cwd=ROOT / "chemkan", capture_output=True, text=True, check=False)
print(result.stdout.strip().splitlines()[-1])

20 passed in 8.11s


## 8. The earlier validation record (development branch, not rerun here)

Before the first FSA training runs, a more extensive validation was run against acceptance
criteria recorded in advance. Its record (`results/experiments/validation/fsa/` on the
development branch: one JSON file per section, the criteria and a README) is **not part of
this repository and was not rerun here**. It used the problem builders of the project's earlier
pipeline (the library's physical-state dynamics with an input normalizer, and earlier hydrogen
Stage-2 models), not the final released-example configurations checked above. The
implementation it validated is the same `chemkan.fsa` module. Its outcome, as recorded:

| section | check | recorded status |
|---|---|---|
| A | analytic problem $\mathrm{d}y/\mathrm{d}t = \theta y$ | PASS |
| B | functional RHS vs dynamics wrapper (float64 and float32) | PASS |
| C | FSA vs direct-autograd gradients over three tolerance levels | PASS after an amendment (originally FAIL) |
| D | augmented-solve state vs state-only solve | PASS |
| E | trajectory independence, batched vs individual solves | PASS after an amendment (originally FAIL) |
| F | float32 gradients vs a float64 reference | PASS |
| G | finite differences (optional) | INCONCLUSIVE |

The amendment kept the thresholds and added one tolerance level finer (1e-12/1e-14): for
hydrogen Stage 1 the gradients of *both* backends were not yet converged at the finest
pre-registered level, and agreed one decade further. Section G passed for four parameters
and was inconclusive for one $\mathrm{KAN_{cor}}$ coefficient, whose gradient (3.7e-8) was too
small for finite differences to resolve. The record also covers resume (bit-identical
continuation), direct-autograd regression and smoke tests, all recorded as PASS.

## 9. What this establishes, and its limits

- The implementation solves the forward sensitivity equations of the stage being trained and
  assembles the gradient by $\sum_j S(t_j)^\top \partial L/\partial x(t_j)$: on the analytic
  problem, state, sensitivity and gradient errors fall with the tolerance (section 2).
- FSA differentiates exactly the right-hand side the trainers integrate (section 3, agreement
  to rounding).
- On the final biodiesel model and hydrogen Stage 1 and Stage 2 with their committed weights,
  FSA and direct-autograd gradients converge to each other as the tolerance tightens
  (sections 4–5); at rtol 1e-10 they agree to about 4e-6 (biodiesel), 3e-6 (hydrogen
  Stage 1) and 2e-7 (hydrogen Stage 2), in every parameter group.
- At the hydrogen trainer's tolerance (rtol 1e-6) the two backends differ by 5e-5 (Stage 1)
  and 3e-4 (Stage 2), the size of their discretization errors. Which backend is closer to the
  converged gradient there depends on the case: FSA for hydrogen Stage 2 (2e-5 vs 3e-4) and
  biodiesel at rtol 1e-6, direct autograd for hydrogen Stage 1 (2e-5 vs 6e-5).
- At the biodiesel training tolerance (rtol 1e-2) the gradients of both backends are far from
  converged (section 5). The recorded biodiesel figure runs trained with FSA at that
  tolerance, as the released example's settings prescribe; their gradients are therefore
  coarse approximations, although the losses they report are accurate.
- Trajectories in a batch are independent (exactly zero cross-derivatives), and batched and
  individual solves agree to the solver's accuracy (section 6).

Limits:

- These checks compare gradients at fixed weights, at one model per stage. They do not show that training with either
  backend reaches the same model: different gradients at the solver's tolerance, amplified over
  many epochs, give different trajectories of the optimization.
- The hydrogen checks use two of the 35 training conditions to keep the runtime short; methane
  uses the same trainer code and was not checked separately here.
- Everything here is float64. The hydrogen and methane runs trained in float32 with direct
  autograd; the float32 comparison exists only in the development-branch record (section 8).
- No finite-difference check is rerun here.
- In this implementation FSA is several times slower per gradient than direct autograd
  (section 4): it forms the Jacobians $\partial F/\partial x$ and $\partial F/\partial\theta$
  explicitly at every right-hand-side evaluation, and the augmented state has $1 + P$ columns.